In [ ]:
# Goal Difference Added (on-pitch + actions), JSON-only
# Definition used here:
# GDA = contribution to goals scored and goals prevented while on the pitch,
#       plus the player's on-ball action value from the event stream.

from pathlib import Path
import json
import math
from datetime import datetime
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

SOURCE_DIR = Path('/Users/marclamberts/Event data/Eredivisie')
OUTPUT_DIR = SOURCE_DIR / 'GDA'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GRID_X_BINS = 12
GRID_Y_BINS = 8
DISCOUNT = 0.97
VALUE_ITERATIONS = 250
MIN_STATE_VISITS = 8
SHOT_TYPE_IDS = {13, 14, 15, 16}
PASS_TYPE_IDS = {1, 2}
CARRY_TAKEON_TYPE_IDS = {3, 4, 5, 6}
DEFENSIVE_TYPE_IDS = {7, 8, 12, 49, 50, 52, 54, 55, 56, 57, 58, 59, 60, 61}

# Opta qualifiers used in these feeds.
Q_PLAYERS = 30
Q_POSITIONS = 44
Q_RELATED_EVENT = 55
Q_SHIRT_NUMBERS = 59
Q_FORMATION_SLOTS = 131
Q_END_X = 140
Q_END_Y = 141
Q_BLOCKED = 82
Q_KEEPER_SAVED_OFF_TARGET = 137
Q_SECOND_YELLOW = 32
Q_RED_CARD = 33


def safe_int(value, default=0):
    try:
        return int(value)
    except Exception:
        return default


def safe_float(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def get_q(event, qid, default=None):
    for q in event.get('qualifier', []) or []:
        if safe_int(q.get('qualifierId'), -1) == qid:
            return q.get('value', default)
    return default


def has_q(event, qid):
    return any(safe_int(q.get('qualifierId'), -1) == qid for q in event.get('qualifier', []) or [])


def split_csv_value(value):
    if value is None:
        return []
    return [part.strip() for part in str(value).split(',')]


def event_sort_key(event, fallback_index=0):
    return (
        safe_int(event.get('periodId')),
        safe_int(event.get('timeMin')),
        safe_int(event.get('timeSec')),
        safe_int(event.get('eventId'), fallback_index),
        fallback_index,
    )


def event_seconds(event):
    return safe_int(event.get('timeMin')) * 60 + safe_int(event.get('timeSec'))


def zone_id(x, y):
    x = float(np.clip(safe_float(x, 50), 0, 100 - 1e-6))
    y = float(np.clip(safe_float(y, 50), 0, 100 - 1e-6))
    xb = int(np.floor(x / 100 * GRID_X_BINS))
    yb = int(np.floor(y / 100 * GRID_Y_BINS))
    return yb * GRID_X_BINS + xb


def zone_center(z):
    xb = z % GRID_X_BINS
    yb = z // GRID_X_BINS
    return ((xb + 0.5) * 100 / GRID_X_BINS, (yb + 0.5) * 100 / GRID_Y_BINS)


def action_category(type_id):
    type_id = safe_int(type_id)
    if type_id in SHOT_TYPE_IDS:
        return 'shot'
    if type_id in PASS_TYPE_IDS:
        return 'pass'
    if type_id in CARRY_TAKEON_TYPE_IDS:
        return 'carry_takeon'
    if type_id in DEFENSIVE_TYPE_IDS:
        return 'defensive'
    return 'other'


def shot_outcome(type_id, event):
    type_id = safe_int(type_id)
    if type_id == 16:
        return 'goal'
    if type_id == 14:
        return 'post'
    if type_id == 15 and has_q(event, Q_BLOCKED):
        return 'blocked'
    if type_id == 15 and has_q(event, Q_KEEPER_SAVED_OFF_TARGET):
        return 'saved_off_target'
    if type_id == 15:
        return 'saved'
    return 'miss'


def load_json(path):
    with path.open('r', encoding='utf-8-sig') as f:
        payload = json.load(f)
    events = payload.get('liveData', {}).get('event', payload.get('event', [])) if isinstance(payload, dict) else []
    return payload, sorted(events if isinstance(events, list) else [], key=lambda e: event_sort_key(e))


def match_duration_seconds(payload, events):
    details = payload.get('matchDetails', {}) if isinstance(payload, dict) else {}
    minutes = safe_int(details.get('matchLengthMin'), 0)
    seconds = safe_int(details.get('matchLengthSec'), 0)
    duration = minutes * 60 + seconds
    if duration <= 0:
        duration = max([event_seconds(e) for e in events if safe_int(e.get('periodId')) in (1, 2)] + [90 * 60])
    return duration


def parse_match_names(path):
    stem = path.stem
    if len(stem) > 11 and ' - ' in stem[11:]:
        home, away = stem[11:].split(' - ', 1)
    else:
        home, away = '', ''
    return stem, home, away


def parse_lineups(events, match_file, match_name, duration_s):
    rows = []
    for e in events:
        if safe_int(e.get('typeId')) != 34:
            continue
        team_id = str(e.get('contestantId', ''))
        players = split_csv_value(get_q(e, Q_PLAYERS, ''))
        slots = [safe_int(v, 0) for v in split_csv_value(get_q(e, Q_FORMATION_SLOTS, ''))]
        shirts = split_csv_value(get_q(e, Q_SHIRT_NUMBERS, ''))
        positions = split_csv_value(get_q(e, Q_POSITIONS, ''))
        max_len = max(len(players), len(slots), len(shirts), len(positions))
        for i in range(max_len):
            player_id = players[i] if i < len(players) else ''
            if not player_id:
                continue
            slot = slots[i] if i < len(slots) else 0
            rows.append({
                'match_file': match_file,
                'match_name': match_name,
                'contestant_id': team_id,
                'player_id': player_id,
                'shirt_number': shirts[i] if i < len(shirts) else '',
                'position_code': positions[i] if i < len(positions) else '',
                'formation_slot': slot,
                'is_starter': int(slot > 0),
                'match_duration_seconds': duration_s,
            })
    return rows


def build_stints(events, lineup_rows, match_file, match_name, duration_s):
    lineups = pd.DataFrame(lineup_rows)
    if lineups.empty:
        return []
    stints = []
    active = {}
    player_team = {}
    for row in lineups.itertuples(index=False):
        player_team[row.player_id] = row.contestant_id
        if int(row.is_starter) == 1:
            active[(row.contestant_id, row.player_id)] = {
                'start_s': 0,
                'start_reason': 'starter',
                'shirt_number': row.shirt_number,
                'position_code': row.position_code,
                'formation_slot': row.formation_slot,
            }

    for e in events:
        type_id = safe_int(e.get('typeId'))
        if type_id not in (18, 19, 17):
            continue
        team_id = str(e.get('contestantId', ''))
        player_id = str(e.get('playerId', ''))
        if not player_id:
            continue
        t = min(max(event_seconds(e), 0), duration_s)
        key = (team_id, player_id)
        if type_id == 18 or (type_id == 17 and (has_q(e, Q_SECOND_YELLOW) or has_q(e, Q_RED_CARD))):
            if key in active:
                stint = active.pop(key)
                if t > stint['start_s']:
                    stints.append({
                        'match_file': match_file,
                        'match_name': match_name,
                        'contestant_id': team_id,
                        'player_id': player_id,
                        'player_name': e.get('playerName', ''),
                        'start_s': stint['start_s'],
                        'end_s': t,
                        'start_reason': stint['start_reason'],
                        'end_reason': 'sub_off' if type_id == 18 else 'red_card',
                        'shirt_number': stint.get('shirt_number', ''),
                        'position_code': stint.get('position_code', ''),
                        'formation_slot': stint.get('formation_slot', 0),
                        'match_duration_seconds': duration_s,
                    })
        elif type_id == 19:
            if key not in active:
                active[key] = {
                    'start_s': t,
                    'start_reason': 'sub_on',
                    'shirt_number': get_q(e, Q_SHIRT_NUMBERS, ''),
                    'position_code': get_q(e, Q_POSITIONS, ''),
                    'formation_slot': 0,
                }

    for (team_id, player_id), stint in active.items():
        if duration_s > stint['start_s']:
            stints.append({
                'match_file': match_file,
                'match_name': match_name,
                'contestant_id': team_id,
                'player_id': player_id,
                'player_name': '',
                'start_s': stint['start_s'],
                'end_s': duration_s,
                'start_reason': stint['start_reason'],
                'end_reason': 'full_time',
                'shirt_number': stint.get('shirt_number', ''),
                'position_code': stint.get('position_code', ''),
                'formation_slot': stint.get('formation_slot', 0),
                'match_duration_seconds': duration_s,
            })
    return stints


json_files = sorted(p for p in SOURCE_DIR.glob('*.json') if p.is_file())
if not json_files:
    raise FileNotFoundError(f'No JSON files found in {SOURCE_DIR}')

all_event_rows = []
all_lineup_rows = []
all_stint_rows = []
all_goal_rows = []
player_names = {}
match_team_names = []

for path in json_files:
    payload, events = load_json(path)
    duration_s = match_duration_seconds(payload, events)
    match_name, home_name, away_name = parse_match_names(path)
    lineup_rows = parse_lineups(events, path.name, match_name, duration_s)
    all_lineup_rows.extend(lineup_rows)
    teams = []
    for row in lineup_rows:
        if row['contestant_id'] not in teams:
            teams.append(row['contestant_id'])
    for i, team_id in enumerate(teams[:2]):
        match_team_names.append({
            'match_file': path.name,
            'contestant_id': team_id,
            'team_name': home_name if i == 0 else away_name,
        })

    all_stint_rows.extend(build_stints(events, lineup_rows, path.name, match_name, duration_s))

    for idx, e in enumerate(events):
        type_id = safe_int(e.get('typeId'), -1)
        team_id = str(e.get('contestantId', ''))
        player_id = str(e.get('playerId', ''))
        player_name = e.get('playerName', '')
        if player_id and player_name:
            player_names[player_id] = player_name
        x = safe_float(e.get('x'))
        y = safe_float(e.get('y'))
        if not team_id or pd.isna(x) or pd.isna(y):
            continue
        end_x = safe_float(get_q(e, Q_END_X))
        end_y = safe_float(get_q(e, Q_END_Y))
        if pd.isna(end_x) or pd.isna(end_y):
            end_x, end_y = x, y
        is_shot = type_id in SHOT_TYPE_IDS
        event_id = str(e.get('id', e.get('eventId', f'{path.name}-{idx}')))
        row = {
            'match_file': path.name,
            'match_name': match_name,
            'event_id': event_id,
            'event_index': safe_int(e.get('eventId'), idx),
            'period_id': safe_int(e.get('periodId')),
            'time_min': safe_int(e.get('timeMin')),
            'time_sec': safe_int(e.get('timeSec')),
            'time_seconds': event_seconds(e),
            'contestant_id': team_id,
            'player_id': player_id,
            'player_name': player_name,
            'type_id': type_id,
            'action_category': action_category(type_id),
            'outcome': safe_int(e.get('outcome'), 0),
            'x': x,
            'y': y,
            'end_x': end_x,
            'end_y': end_y,
            'start_zone': zone_id(x, y),
            'end_zone': zone_id(end_x, end_y),
            'is_shot': int(is_shot),
            'is_goal': int(type_id == 16),
            'shot_outcome': shot_outcome(type_id, e) if is_shot else '',
        }
        all_event_rows.append(row)
        if type_id == 16:
            all_goal_rows.append({
                'match_file': path.name,
                'match_name': match_name,
                'time_seconds': event_seconds(e),
                'contestant_id': team_id,
                'player_id': player_id,
                'player_name': player_name,
                'event_id': event_id,
            })

events_df = pd.DataFrame(all_event_rows).sort_values(['match_file', 'period_id', 'time_min', 'time_sec', 'event_index']).reset_index(drop=True)
lineups_df = pd.DataFrame(all_lineup_rows)
stints_df = pd.DataFrame(all_stint_rows)
goals_df = pd.DataFrame(all_goal_rows)
team_names_df = pd.DataFrame(match_team_names).drop_duplicates() if match_team_names else pd.DataFrame(columns=['match_file', 'contestant_id', 'team_name'])

if events_df.empty or stints_df.empty:
    raise RuntimeError('Could not build events/stints from the JSON files.')

# Fill names for players whose lineup stint did not carry a playerName.
stints_df['player_name'] = stints_df.apply(
    lambda r: r['player_name'] if str(r['player_name']) else player_names.get(r['player_id'], ''), axis=1
)
stints_df['minutes'] = (stints_df['end_s'] - stints_df['start_s']) / 60.0
print(f'Loaded {len(events_df):,} coordinate events, {len(stints_df):,} stints, {len(goals_df):,} goals from {len(json_files):,} matches')

# -----------------------------------------------------------------------------
# JSON-only action value model (for on-ball component)
# -----------------------------------------------------------------------------
shot_rows = events_df[events_df['is_shot'].eq(1)].copy()
global_goal_rate = float(shot_rows['is_goal'].mean()) if len(shot_rows) else 0.10
zone_rates = (
    shot_rows.groupby('start_zone')['is_goal']
    .agg(['sum', 'count'])
    .assign(rate=lambda d: (d['sum'] + global_goal_rate * 20) / (d['count'] + 20))
    ['rate']
    .to_dict()
)
events_df['xg_json'] = np.where(
    events_df['is_shot'].eq(1),
    events_df['start_zone'].map(zone_rates).fillna(global_goal_rate),
    0.0,
)

events_df['next_index'] = events_df.groupby('match_file').cumcount() + 1
next_df = events_df[['match_file', 'period_id', 'contestant_id', 'x', 'y', 'start_zone']].copy()
next_df['next_index'] = next_df.groupby('match_file').cumcount()
next_df = next_df.rename(columns={
    'period_id': 'next_period_id',
    'contestant_id': 'next_contestant_id',
    'x': 'next_x',
    'y': 'next_y',
    'start_zone': 'next_event_zone',
})
events_df = events_df.merge(next_df, on=['match_file', 'next_index'], how='left')
same_period = events_df['period_id'].eq(events_df['next_period_id'])
same_team_next = same_period & events_df['contestant_id'].eq(events_df['next_contestant_id'])
has_next = events_df['next_contestant_id'].notna() & same_period
turnover = has_next & ~same_team_next

events_df['post_sign'] = np.where(same_team_next, 1.0, np.where(has_next, -1.0, 0.0))
events_df['post_zone'] = events_df['end_zone']
if turnover.any():
    fx = 100 - pd.to_numeric(events_df.loc[turnover, 'next_x'], errors='coerce').fillna(50)
    fy = 100 - pd.to_numeric(events_df.loc[turnover, 'next_y'], errors='coerce').fillna(50)
    events_df.loc[turnover, 'post_zone'] = [zone_id(a, b) for a, b in zip(fx, fy)]
shot_reset = events_df['is_shot'].eq(1) & ~events_df['shot_outcome'].eq('blocked')
events_df.loc[shot_reset, 'post_sign'] = -1.0
events_df.loc[shot_reset, 'post_zone'] = zone_id(8, 50)
events_df.loc[events_df['is_goal'].eq(1), 'post_sign'] = 0.0

t = events_df[['start_zone', 'post_zone', 'post_sign', 'xg_json']].copy()
t['post_zone'] = t['post_zone'].fillna(t['start_zone']).astype(int)
state_counts = t.groupby('start_zone').size().to_dict()
n_states = GRID_X_BINS * GRID_Y_BINS
V = np.zeros(n_states, dtype=float)
by_state = {int(k): g for k, g in t.groupby('start_zone')}
for _ in range(VALUE_ITERATIONS):
    new_V = V.copy()
    for state in range(n_states):
        g = by_state.get(state)
        if g is None or len(g) < MIN_STATE_VISITS:
            cx, _ = zone_center(state)
            new_V[state] = 0.015 * (cx / 100.0) ** 2
            continue
        continuation = g['post_sign'].to_numpy(float) * V[g['post_zone'].to_numpy(int)]
        new_V[state] = np.mean(g['xg_json'].to_numpy(float) + DISCOUNT * continuation)
    if np.max(np.abs(new_V - V)) < 1e-7:
        V = new_V
        break
    V = new_V

pre_zone = events_df['start_zone'].astype(int).clip(0, n_states - 1)
post_zone = events_df['post_zone'].fillna(events_df['start_zone']).astype(int).clip(0, n_states - 1)
events_df['action_pre_value'] = V[pre_zone.to_numpy()]
events_df['action_post_value'] = events_df['post_sign'].astype(float).to_numpy() * V[post_zone.to_numpy()]
events_df['action_gda_expected'] = events_df['xg_json'] + DISCOUNT * events_df['action_post_value'] - events_df['action_pre_value']
events_df['action_gda_actual'] = events_df['is_goal'] + DISCOUNT * events_df['action_post_value'] - events_df['action_pre_value']

# -----------------------------------------------------------------------------
# On-pitch score/conceding component
# -----------------------------------------------------------------------------
team_totals = (
    goals_df.groupby(['match_file', 'contestant_id']).size().rename('team_goals').reset_index()
    if not goals_df.empty else pd.DataFrame(columns=['match_file', 'contestant_id', 'team_goals'])
)
match_team_ids = lineups_df[['match_file', 'contestant_id', 'match_duration_seconds']].drop_duplicates()
match_team_ids = match_team_ids.merge(team_totals, on=['match_file', 'contestant_id'], how='left').fillna({'team_goals': 0})
# Opponent goals in the same match.
opp_rows = []
for match_file, g in match_team_ids.groupby('match_file'):
    teams = g['contestant_id'].tolist()
    for _, row in g.iterrows():
        opp_goals = g.loc[g['contestant_id'].ne(row['contestant_id']), 'team_goals'].sum()
        opp_rows.append({'match_file': match_file, 'contestant_id': row['contestant_id'], 'team_goals_against': opp_goals})
match_team_ids = match_team_ids.merge(pd.DataFrame(opp_rows), on=['match_file', 'contestant_id'], how='left')

stint_goal_rows = []
for row in stints_df.itertuples(index=False):
    gmatch = goals_df[goals_df['match_file'].eq(row.match_file)] if not goals_df.empty else pd.DataFrame()
    if gmatch.empty:
        gf = ga = 0
    else:
        in_window = gmatch['time_seconds'].ge(row.start_s) & gmatch['time_seconds'].lt(row.end_s)
        gf = int((in_window & gmatch['contestant_id'].eq(row.contestant_id)).sum())
        ga = int((in_window & gmatch['contestant_id'].ne(row.contestant_id)).sum())
    stint_goal_rows.append({
        'match_file': row.match_file,
        'contestant_id': row.contestant_id,
        'player_id': row.player_id,
        'start_s': row.start_s,
        'end_s': row.end_s,
        'goals_for_on': gf,
        'goals_against_on': ga,
    })
stint_goals_df = pd.DataFrame(stint_goal_rows)
stints_df = stints_df.merge(stint_goals_df, on=['match_file', 'contestant_id', 'player_id', 'start_s', 'end_s'], how='left')
stints_df[['goals_for_on', 'goals_against_on']] = stints_df[['goals_for_on', 'goals_against_on']].fillna(0).astype(int)
stints_df['goal_difference_on'] = stints_df['goals_for_on'] - stints_df['goals_against_on']

player_match = (
    stints_df.groupby(['match_file', 'match_name', 'contestant_id', 'player_id', 'player_name'], dropna=False)
    .agg(
        minutes=('minutes', 'sum'),
        goals_for_on=('goals_for_on', 'sum'),
        goals_against_on=('goals_against_on', 'sum'),
        goal_difference_on=('goal_difference_on', 'sum'),
        stint_count=('player_id', 'count'),
        match_duration_seconds=('match_duration_seconds', 'max'),
    )
    .reset_index()
)
player_match = player_match.merge(
    match_team_ids.drop(columns=['match_duration_seconds'], errors='ignore'),
    on=['match_file', 'contestant_id'],
    how='left',
)
player_match['match_minutes'] = player_match['match_duration_seconds'] / 60.0
player_match['off_minutes'] = (player_match['match_minutes'] - player_match['minutes']).clip(lower=0)
player_match['goals_for_off'] = player_match['team_goals'] - player_match['goals_for_on']
player_match['goals_against_off'] = player_match['team_goals_against'] - player_match['goals_against_on']
player_match['goal_difference_off'] = player_match['goals_for_off'] - player_match['goals_against_off']

player_actions = (
    events_df[events_df['player_id'].astype(str).ne('')]
    .groupby(['player_id'], dropna=False)
    .agg(
        actions=('event_id', 'count'),
        shots=('is_shot', 'sum'),
        goals=('is_goal', 'sum'),
        action_gda_expected=('action_gda_expected', 'sum'),
        action_gda_actual=('action_gda_actual', 'sum'),
    )
    .reset_index()
)

player_summary = (
    player_match.groupby(['player_id', 'player_name', 'contestant_id'], dropna=False)
    .agg(
        matches=('match_file', 'nunique'),
        minutes=('minutes', 'sum'),
        goals_for_on=('goals_for_on', 'sum'),
        goals_against_on=('goals_against_on', 'sum'),
        goal_difference_on=('goal_difference_on', 'sum'),
        goals_for_off=('goals_for_off', 'sum'),
        goals_against_off=('goals_against_off', 'sum'),
        goal_difference_off=('goal_difference_off', 'sum'),
        off_minutes=('off_minutes', 'sum'),
    )
    .reset_index()
)
player_summary['player_name'] = player_summary.apply(
    lambda r: r['player_name'] if str(r['player_name']) else player_names.get(r['player_id'], ''), axis=1
)
player_summary = player_summary.merge(player_actions, on='player_id', how='left').fillna({
    'actions': 0, 'shots': 0, 'goals': 0, 'action_gda_expected': 0.0, 'action_gda_actual': 0.0,
})

player_summary['goals_scored_component'] = player_summary['goals_for_on']
player_summary['goals_prevented_component'] = -player_summary['goals_against_on']
player_summary['score_component'] = player_summary['goal_difference_on']
player_summary['score_component_per90'] = player_summary['score_component'] / player_summary['minutes'].replace(0, np.nan) * 90
player_summary['goals_for_on_per90'] = player_summary['goals_for_on'] / player_summary['minutes'].replace(0, np.nan) * 90
player_summary['goals_against_on_per90'] = player_summary['goals_against_on'] / player_summary['minutes'].replace(0, np.nan) * 90
player_summary['off_goal_difference_per90'] = player_summary['goal_difference_off'] / player_summary['off_minutes'].replace(0, np.nan) * 90
player_summary['relative_on_off_goal_difference_per90'] = player_summary['score_component_per90'] - player_summary['off_goal_difference_per90']
player_summary['action_gda_per90'] = player_summary['action_gda_expected'] / player_summary['minutes'].replace(0, np.nan) * 90
player_summary['goal_difference_added'] = player_summary['score_component'] + player_summary['action_gda_expected']
player_summary['goal_difference_added_per90'] = player_summary['goal_difference_added'] / player_summary['minutes'].replace(0, np.nan) * 90
player_summary['relative_goal_difference_added_per90'] = player_summary['relative_on_off_goal_difference_per90'].fillna(0) + player_summary['action_gda_per90'].fillna(0)
player_summary['actions_per90'] = player_summary['actions'] / player_summary['minutes'].replace(0, np.nan) * 90

player_summary = player_summary.sort_values('goal_difference_added_per90', ascending=False)

team_summary = (
    player_match.groupby(['contestant_id'], dropna=False)
    .agg(
        player_minutes=('minutes', 'sum'),
        player_matches=('player_id', 'count'),
        goals_for_on=('goals_for_on', 'sum'),
        goals_against_on=('goals_against_on', 'sum'),
    )
    .reset_index()
)

zone_values = pd.DataFrame({
    'zone': np.arange(n_states),
    'zone_x': [zone_center(z)[0] for z in range(n_states)],
    'zone_y': [zone_center(z)[1] for z in range(n_states)],
    'state_visits': [state_counts.get(z, 0) for z in range(n_states)],
    'action_state_value': V,
})

# Save outputs.
events_out = events_df[[
    'match_file', 'event_id', 'event_index', 'period_id', 'time_min', 'time_sec', 'time_seconds',
    'contestant_id', 'player_id', 'player_name', 'type_id', 'action_category', 'x', 'y', 'end_x', 'end_y',
    'is_shot', 'is_goal', 'xg_json', 'action_pre_value', 'action_post_value',
    'action_gda_expected', 'action_gda_actual'
]].copy()

stints_df.to_csv(OUTPUT_DIR / 'gda_player_stints.csv', index=False)
player_match.to_csv(OUTPUT_DIR / 'gda_player_match_on_pitch.csv', index=False)
events_out.to_csv(OUTPUT_DIR / 'gda_action_values.csv', index=False)
player_summary.to_csv(OUTPUT_DIR / 'gda_player_summary.csv', index=False)
team_summary.to_csv(OUTPUT_DIR / 'gda_team_summary.csv', index=False)
zone_values.to_csv(OUTPUT_DIR / 'gda_zone_values.csv', index=False)

meta = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'source': str(SOURCE_DIR),
    'definition': 'GDA = on-pitch goals scored minus goals conceded, plus player on-ball action value from JSON event actions.',
    'on_pitch_component': 'goals_for_on - goals_against_on while player stint is active',
    'action_component': 'JSON-only Markov possession value over pitch zones; shot rewards learned from smoothed empirical goal rates by shot zone.',
    'red_card_handling': 'typeId 17 with qualifier 32 or 33 ends the stint at event time',
    'substitution_handling': 'typeId 18 ends stint; typeId 19 starts stint',
    'files': [
        'gda_player_summary.csv', 'gda_player_match_on_pitch.csv', 'gda_player_stints.csv',
        'gda_action_values.csv', 'gda_team_summary.csv', 'gda_zone_values.csv'
    ],
}
with (OUTPUT_DIR / 'gda_model_meta.json').open('w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2)

print('\nSaved on-pitch GDA outputs to:', OUTPUT_DIR)
for name in meta['files'] + ['gda_model_meta.json']:
    path = OUTPUT_DIR / name
    print(f'  {name:32s} {path.stat().st_size / 1024:9.1f} KB')

print('\nTop players by Goal Difference Added per 90 (min 900 minutes):')
cols = [
    'player_name', 'matches', 'minutes', 'goals_for_on', 'goals_against_on',
    'score_component', 'action_gda_expected', 'goal_difference_added',
    'goal_difference_added_per90', 'relative_goal_difference_added_per90'
]
print(player_summary[player_summary['minutes'] >= 900][cols].head(25).round(3).to_string(index=False))
